# Heterogeneous MPNN Experiment on Preprocessed Elliptic++

This notebook only loads the already-preprocessed heterogeneous graph, trains a Heterogeneous MPNN, and reports results.  
No preprocessing, no data splitting, and no visualization.

In [ ]:
# Install dependencies
!pip install -q torch-geometric scikit-learn tqdm

# Sparse kernels (faster message passing — safe to skip if slow)
import subprocess, sys, torch
try:
    v    = torch.__version__.split('+')[0]
    cuda = 'cu121' if torch.cuda.is_available() else 'cpu'
    base = f'https://data.pyg.org/whl/torch-{v}+{cuda}.html'
    for pkg in ['torch_scatter','torch_sparse','torch_cluster']:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','--find-links',base,pkg])
    print('Sparse kernels installed.')
except Exception as e:
    print(f'Sparse kernels skipped: {e}')
print('\n✅ Ready')

Sparse kernels installed.

✅ Ready


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Path to your already-preprocessed graph files
GRAPH_FILE = "/content/drive/MyDrive/EllipticPlusPlus/preprocessed/hetero_graph_data.pt"   # ← your file path
DATA_PATH  = os.path.dirname(GRAPH_FILE)

assert os.path.exists(GRAPH_FILE), f"Graph file not found: {GRAPH_FILE}"
print("✅ File found:", GRAPH_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File found: /content/drive/MyDrive/EllipticPlusPlus/preprocessed/hetero_graph_data.pt


In [ ]:
# Load the preprocessed heterogeneous graph
try:
    data = torch.load(GRAPH_FILE, map_location="cpu", weights_only=False)
except TypeError:
    data = torch.load(GRAPH_FILE, map_location="cpu")

print(data)
print("Node types:", data.node_types)
print("Edge types:", data.edge_types)

HeteroData(
  wallet={
    x=[1268260, 56],
    y=[1268260],
    train_mask=[1268260],
    test_mask=[1268260],
  },
  transaction={
    x=[203769, 182],
    y=[203769],
    train_mask=[203769],
    test_mask=[203769],
  },
  (wallet, sends, transaction)={ edge_index=[2, 477117] },
  (transaction, received_by, wallet)={ edge_index=[2, 837124] },
  (wallet, interacts_with, wallet)={ edge_index=[2, 2868964] },
  (transaction, flows_to, transaction)={ edge_index=[2, 234355] }
)
Node types: ['wallet', 'transaction']
Edge types: [('wallet', 'sends', 'transaction'), ('transaction', 'received_by', 'wallet'), ('wallet', 'interacts_with', 'wallet'), ('transaction', 'flows_to', 'transaction')]


In [ ]:
# # Use the existing 80/20 split from the graph file
# TARGET_NODE = "transaction"

# assert hasattr(data[TARGET_NODE], "x"), f"{TARGET_NODE}.x not found"
# assert hasattr(data[TARGET_NODE], "y"), f"{TARGET_NODE}.y not found"
# assert hasattr(data[TARGET_NODE], "train_mask"), "train_mask not found. The .pt file must already contain the split."
# assert hasattr(data[TARGET_NODE], "test_mask"), "test_mask not found. The .pt file must already contain the split."

# # Fix label dtype — must be 1-D integer tensor
# for nt in data.node_types:
#     data[nt].y = data[nt].y.long().view(-1)

# # Remove -1 (unlabeled) nodes from train/test masks
# for nt in data.node_types:
#     data[nt].train_mask = data[nt].train_mask & (data[nt].y >= 0)
#     data[nt].test_mask  = data[nt].test_mask  & (data[nt].y >= 0)

# num_classes = int(data[TARGET_NODE].y[data[TARGET_NODE].y >= 0].max().item()) + 1

# print("Target node:", TARGET_NODE)
# print("Number of classes:", num_classes)
# print("Train samples:", int(data[TARGET_NODE].train_mask.sum()))
# print("Test samples:", int(data[TARGET_NODE].test_mask.sum()))

# y_labeled = data[TARGET_NODE].y[data[TARGET_NODE].y >= 0].cpu()
# print("Label distribution:", torch.bincount(y_labeled))

# Use the existing split from the graph file
TARGET_NODE = "wallet"   # ← changed: we classify accounts, not transactions

assert hasattr(data[TARGET_NODE], "x"), f"{TARGET_NODE}.x not found"
assert hasattr(data[TARGET_NODE], "y"), f"{TARGET_NODE}.y not found"
assert hasattr(data[TARGET_NODE], "train_mask"), "train_mask not found."
assert hasattr(data[TARGET_NODE], "test_mask"),  "test_mask not found."

# Fix label dtype — must be 1-D integer tensor
for nt in data.node_types:
    data[nt].y = data[nt].y.long().view(-1)

# Remove -1 (unlabelled) nodes from train/test masks
for nt in data.node_types:
    data[nt].train_mask = data[nt].train_mask & (data[nt].y >= 0)
    data[nt].test_mask  = data[nt].test_mask  & (data[nt].y >= 0)

num_classes = int(data[TARGET_NODE].y[data[TARGET_NODE].y >= 0].max().item()) + 1

print("Target node    :", TARGET_NODE)
print("Number of classes:", num_classes)
print("Train samples  :", int(data[TARGET_NODE].train_mask.sum()))
print("Test samples   :", int(data[TARGET_NODE].test_mask.sum()))

y_labeled = data[TARGET_NODE].y[data[TARGET_NODE].y >= 0].cpu()
print("Label distribution:", torch.bincount(y_labeled))

Target node    : wallet
Number of classes: 2
Train samples  : 293977
Test samples   : 73495
Label distribution: tensor([338871,  28601])


In [ ]:
class HeteroMPNN(nn.Module):
    def __init__(self, metadata, hidden_channels, out_channels, dropout=0.3):
        super().__init__()

        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in metadata[1]
        }, aggr="sum")

        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in metadata[1]
        }, aggr="sum")

        self.lin = Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {node_type: F.relu(x) for node_type, x in x_dict.items()}
        x_dict = {
            node_type: F.dropout(x, p=self.dropout, training=self.training)
            for node_type, x in x_dict.items()
        }

        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {node_type: F.relu(x) for node_type, x in x_dict.items()}

        return self.lin(x_dict[TARGET_NODE])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

model = HeteroMPNN(
    metadata=data.metadata(),
    hidden_channels=64,
    out_channels=num_classes,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
# Class weights to handle severe imbalance (illicit is very rare)
y_train  = data[TARGET_NODE].y[data[TARGET_NODE].train_mask].cpu()
counts   = torch.bincount(y_train.long().view(-1), minlength=num_classes).float()
weights  = (1.0 / (counts + 1e-8))
weights  = (weights / weights.sum() * num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print("Device:", device)
print(model)

Device: cpu
HeteroMPNN(
  (conv1): HeteroConv(num_relations=4)
  (conv2): HeteroConv(num_relations=4)
  (lin): Linear(64, 2, bias=True)
)


In [ ]:
def precision_at_recall(y_true, probs, recall_thresholds=[0.01, 0.05, 0.10, 0.50]):
    from sklearn.metrics import precision_recall_curve
    precision, recall, _ = precision_recall_curve(y_true, probs)
    results = {}
    for r in recall_thresholds:
        mask = recall >= r
        if mask.any():
            results[f'P@R{int(r*100)}%'] = float(precision[mask].max())
        else:
            results[f'P@R{int(r*100)}%'] = 0.0
    return results


def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)
    train_mask = data[TARGET_NODE].train_mask

    loss = criterion(out[train_mask], data[TARGET_NODE].y[train_mask].long())
    loss.backward()
    optimizer.step()

    return float(loss.item())


@torch.no_grad()
def evaluate(mask_name):
    model.eval()

    out    = model(data.x_dict, data.edge_index_dict)
    mask   = data[TARGET_NODE][mask_name]
    logits = out[mask]
    y_true = data[TARGET_NODE].y[mask]
    y_pred = logits.argmax(dim=-1)

    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_np, y_pred_np, average="weighted", zero_division=0
    )

    results = {
        "accuracy"             : accuracy_score(y_true_np, y_pred_np),
        "precision_weighted"   : precision,
        "recall_weighted"      : recall,
        "f1_weighted"          : f1,
        "classification_report": classification_report(y_true_np, y_pred_np, zero_division=0),
        "confusion_matrix"     : confusion_matrix(y_true_np, y_pred_np)
    }

    if num_classes == 2:
        probs = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
        results["roc_auc"] = roc_auc_score(y_true_np, probs)
        results["pr_auc"]  = average_precision_score(y_true_np, probs)
        results.update(precision_at_recall(y_true_np, probs))

    return results

In [ ]:
EPOCHS = 100

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch == 1 or epoch % 10 == 0:
        train_results = evaluate("train_mask")
        test_results = evaluate("test_mask")

        pr  = test_results.get("pr_auc",  float("nan"))
        roc = test_results.get("roc_auc", float("nan"))
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train F1: {train_results['f1_weighted']:.4f} | "
            f"Test F1: {test_results['f1_weighted']:.4f} | "
            f"PR-AUC: {pr:.4f} | ROC-AUC: {roc:.4f}"
        )

Epoch 001 | Loss: 0.7251 | Train F1: 0.9000 | Test F1: 0.9020 | PR-AUC: 0.3349 | ROC-AUC: 0.8423
Epoch 010 | Loss: 0.4223 | Train F1: 0.8659 | Test F1: 0.8658 | PR-AUC: 0.4845 | ROC-AUC: 0.9054
Epoch 020 | Loss: 0.3606 | Train F1: 0.8562 | Test F1: 0.8567 | PR-AUC: 0.6336 | ROC-AUC: 0.9319
Epoch 030 | Loss: 0.3301 | Train F1: 0.8634 | Test F1: 0.8637 | PR-AUC: 0.6808 | ROC-AUC: 0.9419
Epoch 040 | Loss: 0.3036 | Train F1: 0.8826 | Test F1: 0.8819 | PR-AUC: 0.7181 | ROC-AUC: 0.9505
Epoch 050 | Loss: 0.2851 | Train F1: 0.8940 | Test F1: 0.8925 | PR-AUC: 0.7432 | ROC-AUC: 0.9561
Epoch 060 | Loss: 0.2671 | Train F1: 0.8988 | Test F1: 0.8970 | PR-AUC: 0.7699 | ROC-AUC: 0.9609
Epoch 070 | Loss: 0.2535 | Train F1: 0.9024 | Test F1: 0.9008 | PR-AUC: 0.7894 | ROC-AUC: 0.9649
Epoch 080 | Loss: 0.2412 | Train F1: 0.9097 | Test F1: 0.9078 | PR-AUC: 0.8074 | ROC-AUC: 0.9679
Epoch 090 | Loss: 0.2331 | Train F1: 0.9139 | Test F1: 0.9114 | PR-AUC: 0.8196 | ROC-AUC: 0.9701
Epoch 100 | Loss: 0.2200 | Tra

In [ ]:
final_results = evaluate("test_mask")

print("Final Test Results")
print("------------------")
for key, value in final_results.items():
    if key not in ["classification_report", "confusion_matrix"]:
        print(f"{key}: {value}")

print("\nClassification Report:")
print(final_results["classification_report"])

print("\nConfusion Matrix:")
print(final_results["confusion_matrix"])

Final Test Results
------------------
accuracy: 0.9036941288523028
precision_weighted: 0.9502173920996989
recall_weighted: 0.9036941288523028
f1_weighted: 0.9183353753446336
roc_auc: 0.9717762598891333
pr_auc: 0.8288750192728376
P@R1%: 1.0
P@R5%: 0.9952456418383518
P@R10%: 0.9952456418383518
P@R50%: 0.9445178335535006

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.95     67775
           1       0.44      0.92      0.60      5720

    accuracy                           0.90     73495
   macro avg       0.72      0.91      0.77     73495
weighted avg       0.95      0.90      0.92     73495


Confusion Matrix:
[[61127  6648]
 [  430  5290]]


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/EllipticPlusPlus/preprocessed/hetero_mpnn_model.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "metadata": data.metadata(),
    "hidden_channels": 64,
    "out_channels": num_classes,
    "target_node": TARGET_NODE
}, MODEL_PATH)

print("Saved model to:", MODEL_PATH)

Saved model to: /content/drive/MyDrive/EllipticPlusPlus/preprocessed/hetero_mpnn_model.pt
